In [1]:
# ============================================================
# TotalSegmentator lung / thoracic cage one-slice overlay
# ------------------------------------------------------------
# 목적:
#   - 기존 전처리된 ct_1mm_lps NIfTI 사용
#   - 모든 환자에 대해 TotalSegmentator 폐 lobe mask 확인
#   - 환자당 대표 slice 1장만 PNG 저장
#   - 같은 이미지 안에 2개 패널 생성
#       왼쪽: CT + TS lung mask
#       오른쪽: CT + TS lung mask + 흉벽 후보 구조
#
# 흉벽 후보:
#   - ribs
#   - sternum
#   - thoracic vertebrae
#   - body outer contour
#
# 주의:
#   - "흉벽" 자체 mask가 아니라 흉벽을 추정할 때 쓸 수 있는 후보 구조임
#   - 원본 CT / 기존 전처리 결과는 수정하지 않음
#   - 이미 생성된 PNG는 스킵
# ============================================================

from pathlib import Path
from types import SimpleNamespace
import subprocess
import shutil
import json
import gc
import traceback

import cv2
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from tqdm import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

CONFIG = {
    # 기존 전처리 결과 root
    # 이 안에 ct_1mm_lps/Normal/*.nii.gz 가 있어야 함
    "preprocess_root": r"E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1",

    "group_name": "Normal",

    # TotalSegmentator 결과 저장 폴더
    "totalseg_out_dir_name": "totalseg_lung_chestwall_1mm",

    # overlay PNG 저장 폴더
    "overlay_out_dir_name": "ts_lung_chestwall_one_slice_overlay_v1",

    # 전체 환자 처리 여부
    "process_all_patients": True,

    # process_all_patients=False일 때만 사용
    "num_patients": 20,
    "random_seed": 42,

    # 문제 환자만 보고 싶으면 여기에 넣기
    # 비워두면 전체 또는 랜덤
    "target_patient_ids": [
        "subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.325580698241281352835338693869",
        "subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.470912100568074901744259213968",
        "subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314",
    ],

    # 이미 PNG 있으면 스킵
    "skip_existing_png": True,

    # 이미 TotalSegmentator 결과 있으면 재사용
    "skip_existing_totalseg": True,

    # True면 기존 TotalSegmentator 결과 폴더 삭제 후 다시 실행
    "overwrite_totalseg": False,

    # TotalSegmentator fast mode
    # 정밀 확인이면 False 추천
    "use_fast_totalseg": False,

    # TotalSegmentator thread 제한
    "nr_thr_resamp": 1,
    "nr_thr_saving": 1,

    # CT window
    "hu_min": -1000,
    "hu_max": 400,

    # body contour용 threshold
    # 몸통 외곽을 대충 보기 위한 값
    "body_hu_threshold": -500,

    # overlay 저장 DPI
    "save_dpi": 180,

    # 화면에 바로 보여줄지
    # 전체 환자 돌릴 때는 False 추천
    "show_inline": False,

    # TotalSegmentator 폐 lobe ROI
    "lung_roi_names": [
        "lung_upper_lobe_left",
        "lung_lower_lobe_left",
        "lung_upper_lobe_right",
        "lung_middle_lobe_right",
        "lung_lower_lobe_right",
    ],

    # 흉벽 후보 ROI
    # 버전에 따라 rib/sternum 이름이 안 맞으면 TotalSegmentator가 실패할 수 있음
    # 실패하면 아래 코드가 lung-only로 한 번 더 시도함
    "chestwall_candidate_roi_names": [
        "sternum",

        "rib_left_1", "rib_left_2", "rib_left_3", "rib_left_4",
        "rib_left_5", "rib_left_6", "rib_left_7", "rib_left_8",
        "rib_left_9", "rib_left_10", "rib_left_11", "rib_left_12",

        "rib_right_1", "rib_right_2", "rib_right_3", "rib_right_4",
        "rib_right_5", "rib_right_6", "rib_right_7", "rib_right_8",
        "rib_right_9", "rib_right_10", "rib_right_11", "rib_right_12",

        "vertebrae_T1", "vertebrae_T2", "vertebrae_T3", "vertebrae_T4",
        "vertebrae_T5", "vertebrae_T6", "vertebrae_T7", "vertebrae_T8",
        "vertebrae_T9", "vertebrae_T10", "vertebrae_T11", "vertebrae_T12",
    ],

    # 흉곽 후보 ROI 때문에 실패하면 폐 lobe만 다시 시도
    "retry_lung_only_if_failed": True,
}

CFG = SimpleNamespace(**CONFIG)

PRE_ROOT = Path(CFG.preprocess_root)
CT_DIR = PRE_ROOT / "ct_1mm_lps" / CFG.group_name
TS_OUT_ROOT = PRE_ROOT / CFG.totalseg_out_dir_name / CFG.group_name
OVERLAY_OUT = PRE_ROOT / CFG.overlay_out_dir_name

TS_OUT_ROOT.mkdir(parents=True, exist_ok=True)
OVERLAY_OUT.mkdir(parents=True, exist_ok=True)

with open(OVERLAY_OUT / "ts_lung_chestwall_overlay_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("PRE_ROOT:", PRE_ROOT)
print("CT_DIR:", CT_DIR)
print("TS_OUT_ROOT:", TS_OUT_ROOT)
print("OVERLAY_OUT:", OVERLAY_OUT)


# ============================================================
# 2. 기본 함수
# ============================================================

def safe_name(name: str) -> str:
    return (
        str(name)
        .replace("\\", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("*", "_")
        .replace("?", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("[", "")
        .replace("]", "")
    )


def find_totalseg_executable():
    for cmd in ["TotalSegmentator", "totalsegmentator"]:
        if shutil.which(cmd) is not None:
            return cmd

    raise RuntimeError(
        "TotalSegmentator 실행 파일을 찾지 못했음. "
        "Jupyter 커널/conda env에서 TotalSegmentator 명령이 잡히는지 확인 필요."
    )


def hu_to_uint8(slice_hu: np.ndarray, hu_min=-1000, hu_max=400):
    x = np.clip(slice_hu.astype(np.float32), hu_min, hu_max)
    x = (x - hu_min) / float(hu_max - hu_min + 1e-8)
    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)
    return x


def find_ct_1mm_files(ct_dir: Path):
    if not ct_dir.exists():
        raise FileNotFoundError(f"ct_1mm_lps 폴더 없음: {ct_dir}")

    files = sorted(ct_dir.glob("*_1mm_lps.nii.gz"))

    rows = []

    for p in files:
        patient_id = p.name.replace("_1mm_lps.nii.gz", "")

        rows.append({
            "patient_id": patient_id,
            "ct_1mm_lps_nii": str(p),
        })

    if len(rows) == 0:
        raise RuntimeError(f"ct_1mm_lps NIfTI 없음: {ct_dir}")

    return pd.DataFrame(rows)


def read_nii_slice_array(path: Path, z: int):
    """
    NIfTI에서 z slice 한 장만 읽음.
    전체 volume을 float32로 올리지 않아서 메모리 절약.
    """

    path = Path(path)

    reader = sitk.ImageFileReader()
    reader.SetFileName(str(path))
    reader.ReadImageInformation()

    size = list(reader.GetSize())  # x, y, z

    if z < 0 or z >= size[2]:
        raise RuntimeError(f"z 범위 오류: z={z}, size={size}, path={path}")

    reader.SetExtractIndex([0, 0, int(z)])
    reader.SetExtractSize([int(size[0]), int(size[1]), 1])

    img_slice = reader.Execute()
    arr = sitk.GetArrayFromImage(img_slice)

    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    return arr


def same_geometry(img_a: sitk.Image, img_b: sitk.Image):
    return (
        img_a.GetSize() == img_b.GetSize()
        and np.allclose(img_a.GetSpacing(), img_b.GetSpacing())
        and np.allclose(img_a.GetOrigin(), img_b.GetOrigin())
        and np.allclose(img_a.GetDirection(), img_b.GetDirection())
    )


def resample_mask_to_reference(mask_img: sitk.Image, ref_img: sitk.Image):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(ref_img)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    return resampler.Execute(mask_img)


def read_mask_as_bool_reference(mask_path: Path, ref_img: sitk.Image):
    mask_img = sitk.ReadImage(str(mask_path))

    if not same_geometry(mask_img, ref_img):
        mask_img = resample_mask_to_reference(mask_img, ref_img)

    arr = sitk.GetArrayFromImage(mask_img) > 0
    return arr


def build_union_mask(mask_dir: Path, roi_names, ref_img: sitk.Image):
    """
    여러 ROI mask를 하나로 합침.
    없는 mask는 missing_names에 기록.
    """

    ref_size = ref_img.GetSize()
    ref_shape = (int(ref_size[2]), int(ref_size[1]), int(ref_size[0]))

    union = np.zeros(ref_shape, dtype=bool)
    used_names = []
    missing_names = []

    for name in roi_names:
        p = mask_dir / f"{name}.nii.gz"

        if not p.exists():
            missing_names.append(name)
            continue

        try:
            arr = read_mask_as_bool_reference(p, ref_img)

            if arr.shape != ref_shape:
                print("[WARN] shape mismatch after resample:", p, arr.shape, ref_shape)
                missing_names.append(name)
                continue

            union |= arr
            used_names.append(name)

        except Exception as e:
            print("[WARN] mask read failed:", p, str(e))
            missing_names.append(name)

    if len(used_names) == 0:
        return None, used_names, missing_names

    return union, used_names, missing_names


def choose_representative_z_from_lung(lung_mask_3d: np.ndarray):
    """
    TS lung mask가 가장 큰 slice 선택.
    """

    area = lung_mask_3d.sum(axis=(1, 2))

    if area.max() <= 0:
        return None, area

    z = int(np.argmax(area))
    return z, area


def keep_largest_component_2d(mask):
    mask = (mask > 0).astype(np.uint8)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if num <= 1:
        return mask.astype(bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    keep_id = int(np.argmax(areas) + 1)

    return (labels == keep_id)


def fill_holes_2d(mask):
    mask = (mask > 0).astype(np.uint8)

    if mask.sum() == 0:
        return mask.astype(bool)

    flood = (mask * 255).copy()
    h, w = flood.shape
    ff_mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(flood, ff_mask, (0, 0), 255)

    flood_inv = cv2.bitwise_not(flood)
    filled = (mask * 255) | flood_inv

    return (filled > 0)


def build_body_contour_mask(slice_hu: np.ndarray, hu_threshold=-500):
    """
    흉벽 후보를 보기 위한 body outer contour 생성.
    정확한 흉벽 segmentation은 아니고, 몸통 외곽 확인용.
    """

    body = (slice_hu > float(hu_threshold)).astype(np.uint8)

    body = cv2.morphologyEx(
        body,
        cv2.MORPH_CLOSE,
        np.ones((9, 9), np.uint8),
    )

    body = cv2.morphologyEx(
        body,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8),
    )

    body = keep_largest_component_2d(body)
    body = fill_holes_2d(body)

    return body.astype(bool)


# ============================================================
# 3. TotalSegmentator 실행
# ============================================================

def has_required_masks(mask_dir: Path, required_names):
    if not mask_dir.exists():
        return False

    for name in required_names:
        if not (mask_dir / f"{name}.nii.gz").exists():
            return False

    return True


def run_totalsegmentator_for_rois(ct_path: Path, out_dir: Path, roi_names, cfg, log_dir: Path):
    """
    TotalSegmentator ROI subset 실행.
    """

    out_dir = Path(out_dir)
    log_dir = Path(log_dir)

    out_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    exe = find_totalseg_executable()

    cmd = [
        exe,
        "-i", str(ct_path),
        "-o", str(out_dir),
        "--nr_thr_resamp", str(int(cfg.nr_thr_resamp)),
        "--nr_thr_saving", str(int(cfg.nr_thr_saving)),
    ]

    if bool(cfg.use_fast_totalseg):
        cmd.append("-f")

    if len(roi_names) > 0:
        cmd += ["--roi_subset"] + list(roi_names)

    print("[TotalSegmentator CMD]")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    (log_dir / "totalseg_stdout.txt").write_text(
        result.stdout or "",
        encoding="utf-8",
        errors="replace",
    )

    (log_dir / "totalseg_stderr.txt").write_text(
        result.stderr or "",
        encoding="utf-8",
        errors="replace",
    )

    (log_dir / "totalseg_cmd.txt").write_text(
        " ".join(cmd),
        encoding="utf-8",
        errors="replace",
    )

    if result.returncode != 0:
        print("[TotalSegmentator ERROR]")
        print((result.stderr or "")[-2000:])
        return False

    return True


def ensure_totalseg_lung_chestwall(ct_path: Path, patient_id: str, cfg):
    """
    폐 lobe + 흉곽 후보 mask를 확보.
    흉곽 후보 ROI 때문에 실패하면 폐 lobe만 fallback 실행.
    """

    mask_dir = TS_OUT_ROOT / patient_id
    log_dir = OVERLAY_OUT / "logs" / patient_id

    lung_names = list(cfg.lung_roi_names)
    chest_names = list(cfg.chestwall_candidate_roi_names)
    all_names = lung_names + chest_names

    if bool(cfg.overwrite_totalseg) and mask_dir.exists():
        shutil.rmtree(mask_dir)

    if bool(cfg.skip_existing_totalseg) and has_required_masks(mask_dir, lung_names):
        print("[TS SKIP] lung masks exist:", patient_id)
        return mask_dir, "existing_or_partial"

    # 먼저 lung + chestwall 후보 전체 실행
    ok = run_totalsegmentator_for_rois(
        ct_path=ct_path,
        out_dir=mask_dir,
        roi_names=all_names,
        cfg=cfg,
        log_dir=log_dir,
    )

    if ok:
        return mask_dir, "lung_chestwall"

    # 실패하면 lung only 재시도
    if bool(cfg.retry_lung_only_if_failed):
        print("[TS RETRY] lung only:", patient_id)

        ok2 = run_totalsegmentator_for_rois(
            ct_path=ct_path,
            out_dir=mask_dir,
            roi_names=lung_names,
            cfg=cfg,
            log_dir=log_dir / "retry_lung_only",
        )

        if ok2:
            return mask_dir, "lung_only_retry"

    return mask_dir, "failed"


# ============================================================
# 4. 시각화 저장
# ============================================================

def save_two_panel_overlay(
    patient_id: str,
    ct_path: Path,
    z: int,
    lung_mask_3d: np.ndarray,
    chestwall_mask_3d,
    used_lung_names,
    used_chest_names,
    missing_chest_names,
    out_path: Path,
    cfg,
):
    ct_slice = read_nii_slice_array(ct_path, z)
    ct_img = hu_to_uint8(ct_slice, cfg.hu_min, cfg.hu_max)

    lung_slice = lung_mask_3d[z] > 0

    if chestwall_mask_3d is None:
        chest_slice = np.zeros_like(lung_slice, dtype=bool)
    else:
        chest_slice = chestwall_mask_3d[z] > 0

    body_mask = build_body_contour_mask(
        slice_hu=ct_slice,
        hu_threshold=cfg.body_hu_threshold,
    )

    lung_area = int(lung_slice.sum())
    chest_area = int(chest_slice.sum())

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # --------------------------------------------------------
    # 왼쪽: TS lung mask
    # --------------------------------------------------------
    axes[0].imshow(ct_img, cmap="gray")

    if lung_area > 0:
        axes[0].contour(
            lung_slice.astype(np.uint8),
            levels=[0.5],
            colors=["yellow"],
            linewidths=1.5,
        )

    axes[0].set_title(
        f"TS lung lobe mask\n"
        f"{patient_id}\n"
        f"z={z}, lung area={lung_area}",
        fontsize=10,
    )
    axes[0].axis("off")

    # --------------------------------------------------------
    # 오른쪽: 흉벽 후보 구조
    # --------------------------------------------------------
    axes[1].imshow(ct_img, cmap="gray")

    # body outer contour
    if body_mask.sum() > 0:
        axes[1].contour(
            body_mask.astype(np.uint8),
            levels=[0.5],
            colors=["lime"],
            linewidths=1.0,
        )

    # lung contour
    if lung_area > 0:
        axes[1].contour(
            lung_slice.astype(np.uint8),
            levels=[0.5],
            colors=["cyan"],
            linewidths=1.2,
        )

    # ribs/sternum/vertebrae candidates
    if chest_area > 0:
        axes[1].contour(
            chest_slice.astype(np.uint8),
            levels=[0.5],
            colors=["red"],
            linewidths=1.1,
        )

    axes[1].set_title(
        f"Chest-wall candidates\n"
        f"lime=body, cyan=lung, red=rib/sternum/vertebrae\n"
        f"chest area={chest_area}, used chest ROI={len(used_chest_names)}",
        fontsize=9,
    )
    axes[1].axis("off")

    # 안내 텍스트
    fig.suptitle(
        f"Used lung ROI: {len(used_lung_names)} | "
        f"Used chest ROI: {len(used_chest_names)} | "
        f"Missing chest ROI: {len(missing_chest_names)}",
        fontsize=10,
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=int(cfg.save_dpi), bbox_inches="tight")

    if bool(cfg.show_inline):
        plt.show()
    else:
        plt.close(fig)

    return {
        "lung_area": lung_area,
        "chest_area": chest_area,
        "used_lung_count": len(used_lung_names),
        "used_chest_count": len(used_chest_names),
        "missing_chest_count": len(missing_chest_names),
    }


# ============================================================
# 5. 실행
# ============================================================

ct_df = find_ct_1mm_files(CT_DIR)

target_ids = list(getattr(CFG, "target_patient_ids", []))

if len(target_ids) > 0:
    selected_df = ct_df[ct_df["patient_id"].astype(str).isin([str(x) for x in target_ids])].copy()

    if len(selected_df) == 0:
        raise RuntimeError("target_patient_ids에 해당하는 환자를 찾지 못함.")
else:
    if bool(CFG.process_all_patients):
        selected_df = ct_df.copy()
    else:
        selected_df = ct_df.sample(
            n=min(int(CFG.num_patients), len(ct_df)),
            random_state=int(CFG.random_seed),
        ).copy()

print("total ct files:", len(ct_df))
print("selected patients:", len(selected_df))
display(selected_df.head())

summary_rows = []
summary_csv = OVERLAY_OUT / "ts_lung_chestwall_one_slice_summary.csv"

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="TS lung/chestwall overlay", ncols=100, ascii=True):
    patient_id = str(row["patient_id"])
    ct_path = Path(row["ct_1mm_lps_nii"])

    out_path = OVERLAY_OUT / f"{safe_name(patient_id)}_ts_lung_chestwall_overlay.png"

    if bool(CFG.skip_existing_png) and out_path.exists():
        print("[SKIP PNG]", out_path)

        summary_rows.append({
            "patient_id": patient_id,
            "ct_1mm_lps_nii": str(ct_path),
            "status": "skipped_existing_png",
            "overlay_png": str(out_path),
        })

        continue

    try:
        print("\n" + "=" * 100)
        print("patient:", patient_id)
        print("ct:", ct_path)

        ref_img = sitk.ReadImage(str(ct_path))

        mask_dir, ts_status = ensure_totalseg_lung_chestwall(
            ct_path=ct_path,
            patient_id=patient_id,
            cfg=CFG,
        )

        if ts_status == "failed":
            summary_rows.append({
                "patient_id": patient_id,
                "ct_1mm_lps_nii": str(ct_path),
                "status": "totalseg_failed",
                "overlay_png": "",
            })
            continue

        lung_union, used_lung, missing_lung = build_union_mask(
            mask_dir=mask_dir,
            roi_names=CFG.lung_roi_names,
            ref_img=ref_img,
        )

        if lung_union is None:
            print("[SKIP] TS lung mask 없음:", patient_id)

            summary_rows.append({
                "patient_id": patient_id,
                "ct_1mm_lps_nii": str(ct_path),
                "status": "missing_ts_lung",
                "overlay_png": "",
                "ts_status": ts_status,
            })

            continue

        chest_union, used_chest, missing_chest = build_union_mask(
            mask_dir=mask_dir,
            roi_names=CFG.chestwall_candidate_roi_names,
            ref_img=ref_img,
        )

        z, lung_area_per_slice = choose_representative_z_from_lung(lung_union)

        if z is None:
            print("[SKIP] lung area 0:", patient_id)

            summary_rows.append({
                "patient_id": patient_id,
                "ct_1mm_lps_nii": str(ct_path),
                "status": "empty_lung_mask",
                "overlay_png": "",
                "ts_status": ts_status,
            })

            continue

        result_info = save_two_panel_overlay(
            patient_id=patient_id,
            ct_path=ct_path,
            z=int(z),
            lung_mask_3d=lung_union,
            chestwall_mask_3d=chest_union,
            used_lung_names=used_lung,
            used_chest_names=used_chest,
            missing_chest_names=missing_chest,
            out_path=out_path,
            cfg=CFG,
        )

        summary_rows.append({
            "patient_id": patient_id,
            "ct_1mm_lps_nii": str(ct_path),
            "status": "saved",
            "ts_status": ts_status,
            "representative_z": int(z),
            "overlay_png": str(out_path),
            "lung_area": int(result_info["lung_area"]),
            "chest_area": int(result_info["chest_area"]),
            "used_lung_count": int(result_info["used_lung_count"]),
            "used_chest_count": int(result_info["used_chest_count"]),
            "missing_chest_count": int(result_info["missing_chest_count"]),
            "used_lung_names": "|".join(used_lung),
            "used_chest_names": "|".join(used_chest),
            "missing_lung_names": "|".join(missing_lung),
            "missing_chest_names": "|".join(missing_chest),
        })

        print("[SAVED]", out_path)

    except Exception as e:
        print("[ERROR]", patient_id, str(e))
        traceback.print_exc()

        summary_rows.append({
            "patient_id": patient_id,
            "ct_1mm_lps_nii": str(ct_path),
            "status": "error",
            "error": str(e),
            "overlay_png": "",
        })

    finally:
        # 환자 하나 끝날 때 메모리 정리
        for name in [
            "ref_img",
            "lung_union",
            "chest_union",
            "lung_area_per_slice",
        ]:
            if name in locals():
                del locals()[name]

        gc.collect()

    # 중간 저장
    pd.DataFrame(summary_rows).to_csv(
        summary_csv,
        index=False,
        encoding="utf-8-sig",
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n========== FINISHED ==========")
print("summary csv:", summary_csv)

if len(summary_df) > 0:
    display(summary_df["status"].value_counts())
    display(summary_df.head())

PRE_ROOT: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1
CT_DIR: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ct_1mm_lps\Normal
TS_OUT_ROOT: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\totalseg_lung_chestwall_1mm\Normal
OVERLAY_OUT: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ts_lung_chestwall_one_slice_overlay_v1
total ct files: 362
selected patients: 3


,patient_id,ct_1mm_lps_nii
149,subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.3255...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...
150,subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.4709...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...
171,subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.2640...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...


TS lung/chestwall overlay:   0%|                                              | 0/3 [00:00<?, ?it/s]


patient: subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.325580698241281352835338693869
ct: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ct_1mm_lps\Normal\subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.325580698241281352835338693869_1mm_lps.nii.gz
[TotalSegmentator CMD]
TotalSegmentator -i E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ct_1mm_lps\Normal\subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.325580698241281352835338693869_1mm_lps.nii.gz -o E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\totalseg_lung_chestwall_1mm\Normal\subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.325580698241281352835338693869 --nr_thr_resamp 1 --nr_thr_saving 1 --roi_subset lung_upper_lobe_left lung_lower_lobe_left lung_upper_lobe_right lung_middle_lobe_right lung_lower_lobe_right sternum rib_left_1 rib_left_2 rib_left_3 rib_left_4 rib_left_5 rib_left_6 rib_left_7 rib_left_8 rib_left_9 rib_left_10 rib_left_11 rib_left_12 rib_right_1 rib_right_2 rib_right_3 

TS lung/chestwall overlay:  33%|############3                        | 1/3 [02:30<05:01, 150.94s/it]

...

[SAVED] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ts_lung_chestwall_one_slice_overlay_v1\subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.470912100568074901744259213968_ts_lung_chestwall_overlay.png

patient: subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314
ct: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ct_1mm_lps\Normal\subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314_1mm_lps.nii.gz
[TotalSegmentator CMD]
TotalSegmentator -i E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ct_1mm_lps\Normal\subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314_1mm_lps.nii.gz -o E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\totalseg_lung_chestwall_1mm\Normal\subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314 --nr_thr_resamp 1 --nr_thr_saving 1 --roi_subset lung_upper_lobe_left lung_lower_lobe_left lung_upper_lobe_right lung

TS lung/chestwall overlay: 100%|#####################################| 3/3 [07:46<00:00, 155.41s/it]

[SAVED] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ts_lung_chestwall_one_slice_overlay_v1\subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.264090899378396711987322794314_ts_lung_chestwall_overlay.png

========== FINISHED ==========
summary csv: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\ts_lung_chestwall_one_slice_overlay_v1\ts_lung_chestwall_one_slice_summary.csv


status
saved    3
Name: count, dtype: int64

,patient_id,ct_1mm_lps_nii,status,ts_status,representative_z,overlay_png,lung_area,chest_area,used_lung_count,used_chest_count,missing_chest_count,used_lung_names,used_chest_names,missing_lung_names,missing_chest_names
0,subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.3255...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,saved,lung_chestwall,137,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,54433,7641,5,37,0,lung_upper_lobe_left|lung_lower_lobe_left|lung...,sternum|rib_left_1|rib_left_2|rib_left_3|rib_l...,,
1,subset2_1.3.6.1.4.1.14519.5.2.1.6279.6001.4709...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,saved,lung_chestwall,154,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,35075,3964,5,37,0,lung_upper_lobe_left|lung_lower_lobe_left|lung...,sternum|rib_left_1|rib_left_2|rib_left_3|rib_l...,,
2,subset3_1.3.6.1.4.1.14519.5.2.1.6279.6001.2640...,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,saved,lung_chestwall,110,E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA...,35763,3097,5,37,0,lung_upper_lobe_left|lung_lower_lobe_left|lung...,sternum|rib_left_1|rib_left_2|rib_left_3|rib_l...,,
